First you need to run train.py with --config-name=config_toy_example_stacked_flow.yaml

In [ ]:
import numpy as np
import paderbox as pb
import torch
import pandas as pd
import plotly.express as px

import lazy_dataset

from padertorch.base import Model
from pathlib import Path
from train import make_stacked_moons

%matplotlib inline

import plotly.io as pio
pio.renderers.default = "notebook"

# load model

In [ ]:
storage_dir = # Path("path/to/model")

model_dict = pb.io.load_yaml(storage_dir / "config.yaml")

model = Model.from_config(model_dict['trainer']['model'])
cp = torch.load(
    storage_dir / "checkpoints/ckpt_latest.pth",
    map_location=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    weights_only=False
)

# remove speaker encoder
model_weights = cp.copy()
model.load_state_dict(model_weights['model'])
model.eval()

# helper function

In [ ]:
def plotting_fkt(features, high_level_labels, low_level_labels, cut=None):
    # === Helper: One-Hot → Class Index ===
    def to_class_labels(labels):
        labels = np.asarray(labels)
        if labels.ndim > 1:
            return labels.argmax(axis=1)
        return labels

    high_level_cls = to_class_labels(high_level_labels)
    low_level_cls  = to_class_labels(low_level_labels)

    # === Plot 1: high_level Labels ===
    fig_high_level = px.scatter(
        x=features[:, 0],
        y=features[:, 1],
        color=pd.Series(high_level_cls.astype(str)),
        title="Flow Outputs colored by high_level Label",
        labels={'color': 'high_level Label', 'x': 'z1', 'y': 'z2'},
        width=800,
        height=600
    )

    if cut is not None:
        fig_high_level.update_xaxes(range=[-cut, cut])
        fig_high_level.update_yaxes(range=[-cut, cut])

    fig_high_level.show()

    # === Plot 2: low_level Labels ===
    fig_low_level = px.scatter(
        x=features[:, 0],
        y=features[:, 1],
        color=pd.Series(low_level_cls.astype(str)),
        title="Flow Outputs colored by low_level Label",
        labels={'color': 'low_level Label', 'x': 'z1', 'y': 'z2'},
        width=800,
        height=600
    )

    if cut is not None:
        fig_low_level.update_xaxes(range=[-cut, cut])
        fig_low_level.update_yaxes(range=[-cut, cut])

    fig_low_level.show()


In [ ]:
def prepare_example(example):
    observation = np.load(example['observation'])
    example['observation'] = observation.tolist()
    return example

dataset_dict = pb.io.load_json(Path("dataset/stacked_moons/") / "dataset.json")
ds = lazy_dataset.from_dict(dataset_dict['eval'])
ds = ds.map(prepare_example)

# dataset

In [ ]:
x_list = []
high_level_labels = []
low_level_labels = []
for example in ds:
    x_list.append(example['observation'])
    high_level_labels.append(example['high_level'])
    low_level_labels.append(example['low_level'])
x_array = np.asarray(x_list)
high_level_labels = np.asarray(high_level_labels)
low_level_labels = np.asarray(low_level_labels)

plotting_fkt(x_array, high_level_labels, low_level_labels)

# apply forward transformation

In [ ]:
outputs = []
outputs_bottle_neck = []
high_level_labels = []
low_level_labels = []
for example in ds.random_choice(500, replace=False):
    x = torch.tensor(example['observation'])[None,:]
    labels = {
        0: torch.tensor(example['high_level'])[None, None].float(),
        1: torch.tensor(example['low_level'])[None, None].float()
    }
    with torch.no_grad():
        output, y, _, _ = model.forward((x, labels))
    outputs.append(output.tolist())
    outputs_bottle_neck.append(y.tolist())

    high_level_labels.append(example['high_level'])
    low_level_labels.append(example['low_level'])
    
outputs = np.asarray(outputs).squeeze()
outputs_bottle_neck = np.asarray(outputs_bottle_neck).squeeze()

# bottleneck t=0.5

In [ ]:
plotting_fkt(outputs_bottle_neck, high_level_labels, low_level_labels)

# t=1

In [ ]:
plotting_fkt(outputs, high_level_labels, low_level_labels)

# Generation step, sample from a normal distribution

In [ ]:
base_mean = torch.tensor([0,0]).float()
base_std = torch.tensor([[1,0],[0,1]]).float()
base = np.random.multivariate_normal(base_mean, base_std, 500)


high_level_label_list = []
low_level_label_list = []

outputs_generated_list = []
outputs_generated_bottleneck_list = []

for high_level_class_idx in [0,1,2]:
    for low_level_class_idx in [0,1]:
        class_1_high_level_labels = torch.ones(base.shape[0])[:, None] * high_level_class_idx
        class_1_high_level_labels = torch.nn.functional.one_hot(class_1_high_level_labels.long(), num_classes=3).float()
        class_1_low_level_labels = torch.ones(base.shape[0])[:, None] * low_level_class_idx
        
        with torch.no_grad():
            generated_samples, generated_bottleneck, _  = model.sample(
                (
                    torch.tensor(base).float(), 
                    {
                        0: class_1_high_level_labels,
                        1: class_1_low_level_labels
                    }
                ), 
            )

        outputs_generated_list.append(generated_samples)
        outputs_generated_bottleneck_list.append(generated_bottleneck)
        high_level_label_list.append(class_1_high_level_labels)
        low_level_label_list.append(class_1_low_level_labels)

outputs_generated_list = [element for example in outputs_generated_list for element in example]
outputs_generated_list = np.asarray(outputs_generated_list).squeeze()

outputs_generated_bottleneck_list = [element for example in outputs_generated_bottleneck_list for element in example]
outputs_generated_bottleneck_list = np.asarray(outputs_generated_bottleneck_list).squeeze()

high_level_label_list = [element for example in high_level_label_list for element in example]
low_level_label_list = [element for example in low_level_label_list for element in example]

# t = 0.5

In [ ]:
plotting_fkt(outputs_generated_bottleneck_list, np.asarray(high_level_label_list).squeeze(1), np.asarray(low_level_label_list).squeeze(1))

# t = 0

In [ ]:
plotting_fkt(outputs_generated_list, np.asarray(high_level_label_list).squeeze(1), np.asarray(low_level_label_list).squeeze(1))